# Ecological claim support — paper exports

Runs `export_paper_ecological.py`, writes **`data/sdm_predictions/paper_results/`**, then previews CSVs below.

Artifacts:
- **paper_model_performance_table.csv**
- **paper_selected_variables_table.csv**
- **paper_shap_top5_table.csv**
- **paper_habitat_claim_statistics.csv**
- **paper_claim_support_matrix.csv**


In [1]:
from pathlib import Path

import pandas as pd

def repo_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(12):
        if (p / "config.yml").exists():
            return p
        p = p.parent
    raise SystemExit("Could not find config.yml")


REPO = repo_root()
OUT = REPO / "data/sdm_predictions/paper_results"
print("REPO:", REPO)


REPO: /Users/work/Data Science/sheffield-bats


In [2]:
# Regenerate CSVs (~10–30s)
import subprocess
import sys

script = REPO / "notebooks" / "postprocessing" / "export_paper_ecological.py"
subprocess.run([sys.executable, str(script)], cwd=REPO, check=True)
print("Export complete")


performance -> /Users/work/Data Science/sheffield-bats/data/sdm_predictions/paper_results/paper_model_performance_table.csv
variables -> /Users/work/Data Science/sheffield-bats/data/sdm_predictions/paper_results/paper_selected_variables_table.csv
shap -> /Users/work/Data Science/sheffield-bats/data/sdm_predictions/paper_results/paper_shap_top5_table.csv
habitat -> /Users/work/Data Science/sheffield-bats/data/sdm_predictions/paper_results/paper_habitat_claim_statistics.csv
matrix -> /Users/work/Data Science/sheffield-bats/data/sdm_predictions/paper_results/paper_claim_support_matrix.csv
Export complete


In [3]:
def show_csv(name: str, n: int = 25) -> None:
    p = OUT / name
    df = pd.read_csv(p)
    print(f"\n=== {name} ({len(df)} rows) ===")
    print(df.head(n).to_string())
    if len(df) > n:
        print(f"... ({len(df) - n} more rows)")


for fn in [
    "paper_model_performance_table.csv",
    "paper_shap_top5_table.csv",
    "paper_claim_support_matrix.csv",
]:
    show_csv(fn, n=22)



=== paper_model_performance_table.csv (17 rows) ===
                               model_id           common_name                 latin_name activity_type  n_presence  mean_cv_auc  std_cv_auc  threshold  omission_rate_at_presences  suitable_area_percent_of_valid_pixels                caution_flag
0            nyctalus_noctula_in_flight               Noctule           Nyctalus noctula     In flight         312     0.834915    0.070300   0.211836                    0.102564                              37.193837                         NaN
1                nyctalus_noctula_roost               Noctule           Nyctalus noctula         Roost          74     0.811137    0.045454   0.252342                    0.108108                              32.714589                         NaN
2   pipistrellus_pipistrellus_in_flight    Common pipistrelle  Pipistrellus pipistrellus     In flight         630     0.838518    0.067742   0.264156                    0.100000                              3

In [4]:
show_csv("paper_selected_variables_table.csv", n=35)



=== paper_selected_variables_table.csv (218 rows) ===
                        model_id                                 variable           variable_group                                                label                                                                                                                                                          interpretation_description
0          myotis_brandtii_roost        os_distance_distance_to_buildings  distance_or_built_cover                                Distance to buildings                                                                                                                                 Straight-line distance to the nearest building (m).
1          myotis_brandtii_roost                    climate_bioclim_bio_9                  climate                    Mean temperature (driest quarter)                                                                                                                     WorldClim Bio

In [5]:
# Habitat stratifications (truncated preview)
h = pd.read_csv(OUT / "paper_habitat_claim_statistics.csv")
print("Habitat statistic rows:", len(h))
for cid in sorted(h["paper_claim_id"].dropna().unique()):
    sub = h[h["paper_claim_id"] == cid]
    print(f"\n--- {cid} ({len(sub)} rows) ---")
    cols = [c for c in sub.columns if sub[c].notna().any()]
    print(sub[cols].head(12).to_string())


Habitat statistic rows: 54

--- C_brandt_whisker_elevation_valley (30 rows) ---
                       paper_claim_id                 model_id                     analysis         stratifier  bin_index    strat_lo    strat_hi  n_pixels  area_km2_approx  mean_suitability  median_suitability  frac_pixels_above_threshold
20  C_brandt_whisker_elevation_valley    myotis_brandtii_roost               elevation_bins        terrain_dtm        1.0   -1.375026   17.167108    275455          2754.55          0.089497            0.001866                     0.053639
21  C_brandt_whisker_elevation_valley    myotis_brandtii_roost               elevation_bins        terrain_dtm        2.0   17.167108   54.549137    275454          2754.54          0.062527            0.000658                     0.036039
22  C_brandt_whisker_elevation_valley    myotis_brandtii_roost               elevation_bins        terrain_dtm        3.0   54.549137  121.339735    275454          2754.54          0.098753          

Tables on disk (`data/sdm_predictions/paper_results/`) mirror this preview. Raster summaries use quantile strata on the aligned **100 m EPSG:27700** grid; moor proxy combines upland-heath fraction and DTM ≥ study-area p66 thresholds.
